# Assessment 2: Machine Learning and Real-Time Streaming

**Student ID:** 35415169
**Unit:** ITO5202 – Data Processing for Big Data

## Part A: Machine Learning Pipeline

This assessment reuses the airline on-time performance dataset from Assessment 1. The dataset will first be divided into a historical training subset and a fully held-out streaming subset so that records replayed through Kafka in Part B have not been used during model training.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Assessment2_ML_Streaming")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.5


In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
) 

airline_schema = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY_OF_MONTH", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("FL_DATE", StringType(), True),
    StructField("OP_UNIQUE_CARRIER", StringType(), True),
    StructField("TAIL_NUM", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("CRS_DEP_TIME", IntegerType(), True),
    StructField("DEP_TIME", IntegerType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("DEP_DELAY_NEW", DoubleType(), True),
    StructField("DEP_DEL15", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", IntegerType(), True),
    StructField("ARR_TIME", IntegerType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("ARR_DELAY_NEW", DoubleType(), True),
    StructField("ARR_DEL15", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("FLIGHTS", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True)
])
    

In [3]:
file_path = "../T_ONTIME_REPORTING.csv"


df = (
    spark.read
    .option("header", True)
    .schema(airline_schema)
    .csv(file_path)
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


### Target preparation

The machine learning task is binary classification using 'ARR_DEL15' as the target variable. Records with a missing target are excluded because cancelled or diverted flights may not have a valid arrival-delay outcome.

In [5]:
ml_df = df.filter(F.col("ARR_DEL15").isNotNull())
print("Rows with valid ARR_DEL15 target:", ml_df.count())

Rows with valid ARR_DEL15 target: 522269


In [6]:
ml_df.groupBy("ARR_DEL15").count().orderBy("ARR_DEL15").show()

+---------+------+
|ARR_DEL15| count|
+---------+------+
|      0.0|424139|
|      1.0| 98130|
+---------+------+



### Train/stream split

A reproducible 70/30 split is applied using a fixed random seed of 42. The 70% subset is reserved for machine learning model development and evaluation, while the remaining 30% is held out entirely for the simulated real-time streaming stage in Part B.

In [7]:
train_df, stream_df = ml_df.randomSplit([0.7, 0.3], seed=42)

train_count = train_df.count()
stream_count = stream_df.count()

print("Training subset:", train_count)
print("Streaming subset:", stream_count)
print("Total:", train_count + stream_count)

Training subset: 365710
Streaming subset: 156559
Total: 522269


In [8]:
total_count = train_count + stream_count

train_pct = train_count / total_count * 100
stream_pct = stream_count / total_count * 100

print("Training percentage:", round(train_pct, 2), "%")
print("Streaming percentage:", round(stream_pct, 2), "%")

Training percentage: 70.02 %
Streaming percentage: 29.98 %


In [9]:
stream_output_path = "data/stream_data.parquet"

stream_df.write.mode("overwrite").parquet(stream_output_path)

print("streaming subset saved to:", stream_output_path)

streaming subset saved to: data/stream_data.parquet


In [10]:
stream_check = spark.read.parquet(stream_output_path)

print("Reloaded streaming rows:", stream_check.count())

Reloaded streaming rows: 156559


## Feature Selection and Leakage Prevention

The target variable for the classification task is 'ARR_DEL15', which indicates whether a flight arrived at least 15 minutes late. Only features that would be available at, or shortly after, departure are considered for training. Variables that directly describe the final arrival outcome are excluded to avoid data leakage.

Candidate predictors include:

- 'MONTH'
- 'DAY_OF_WEEK'
- 'OP_UNIQUE_CARRIER'
- 'ORIGIN'
- 'DEST'
- 'CRS_DEP_TIME'
- 'CRS_ARR_TIME'
- 'DISTANCE'
- 'DEP_DELAY'

The following variables are excluded because they directly reveal, or are strongly derived from , the final arrival outcome:

- `ARR_DELAY`
- 'ARR_DELAY_NEW'
- 'ARR_DEL15'
- 'ARR_TIME'
- 'ACTUAL_ELAPSED_TIME'
- 'AIR_TIME'
- 'TAXI_IN'

This helps ensure that the model learns from information that would realistically be available before the final arrival outcome is known.

In [12]:
categorical_features = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST"
]


numeric_features = [
    "MONTH",
    "DAY_OF_WEEK",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "DISTANCE",
    "DEP_DELAY"
]

target_col = "ARR_DEL15"

selected_columns = categorical_features + numeric_features + [target_col]

ml_selected_df = train_df.select(*selected_columns)

ml_selected_df.show(5, truncate=False)

+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|MONTH|DAY_OF_WEEK|CRS_DEP_TIME|CRS_ARR_TIME|DISTANCE|DEP_DELAY|ARR_DEL15|
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|AA               |JFK   |SFO |1    |3          |600         |940         |2586.0  |-3.0     |0.0      |
|AA               |SFO   |JFK |1    |3          |1030        |1912        |2586.0  |-6.0     |0.0      |
|AA               |LAX   |JFK |1    |3          |1130        |2001        |2475.0  |41.0     |0.0      |
|AA               |JFK   |LAX |1    |3          |2100        |29          |2475.0  |-8.0     |0.0      |
|AA               |BOS   |LAX |1    |3          |801         |1140        |2611.0  |-5.0     |0.0      |
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
only showing top 5 rows



In [13]:
missing_counts = ml_selected_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ml_selected_df.columns
])

missing_counts.show(truncate=False)

+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|MONTH|DAY_OF_WEEK|CRS_DEP_TIME|CRS_ARR_TIME|DISTANCE|DEP_DELAY|ARR_DEL15|
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|0                |0     |0   |0    |0          |0           |0           |0       |0        |0        |
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+



### Feature Engineering Pipeline

Categorical variables are converted to numerical representations using 'StringIndexer' followed by 'OneHotEncoder'. The encoded categorical variables and numeric predictors are then combined using 'VectorAssembler' into a single 'features' vector required by Spark ML models.

A null-value check confirmed that the selected modelling columns contain no missing values, so additional imputation is not required.



In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_INDEX",
        handleInvalid="keep"
    )
    for col in categorical_features
]

encoder = OneHotEncoder(
    inputCols=[f"{col}_INDEX" for col in categorical_features],
        outputCols=[f"{col}_OHE" for col in categorical_features]
)

assembler = VectorAssembler(
    inputCols=[
        *[f"{col}_OHE" for col in categorical_features],
        *numeric_features
    ],
    outputCol="features"
    
)

preprocessing_pipeline = Pipeline(
    stages=indexers + [encoder, assembler]
    
)
    



In [15]:
preprocessing_model = preprocessing_pipeline.fit(ml_selected_df)

transformed_df = preprocessing_model.transform(ml_selected_df)

transformed_df.select(
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "features",
    "ARR_DEL15"
).show(5,truncate=False)

+-----------------+------+----+----------------------------------------------------------------------------------------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|features                                                                                |ARR_DEL15|
+-----------------+------+----+----------------------------------------------------------------------------------------+---------+
|AA               |JFK   |SFO |(678,[2,34,355,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,600.0,940.0,2586.0,-3.0])  |0.0      |
|AA               |SFO   |JFK |(678,[2,24,363,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,1030.0,1912.0,2586.0,-6.0])|0.0      |
|AA               |LAX   |JFK |(678,[2,21,363,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,1130.0,2001.0,2475.0,41.0])|0.0      |
|AA               |JFK   |LAX |(678,[2,34,349,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,2100.0,29.0,2475.0,-8.0])  |0.0      |
|AA               |BOS   |LAX |(678,[2,27,349,672,673,674,675,676,677],[1.0,1.0,1.0